# Customer Segmentation & Churn Pattern Analytics in European Banking

## Comprehensive Analysis Notebook

This notebook provides a complete analysis of customer churn patterns across European banking, including:
- Data loading and validation
- Exploratory Data Analysis (EDA)
- Customer Segmentation
- Churn Prediction Modeling
- Key Insights and Recommendations

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.data.loader import load_raw_data, save_processed_data
from src.data.preprocessing import DataPreprocessor
from src.features.engineering import FeatureEngineer
from src.models.segmentation import CustomerSegmentation, perform_optimal_clustering
from src.models.churn_model import ChurnPredictor, compare_models
from src.analysis.churn_analytics import ChurnAnalytics
from src.visualization.plotting import ChurnVisualizer
from config.settings import *

print("All libraries imported successfully!")

## 1. Data Loading & Validation

In [ ]:
# Load raw data
df = load_raw_data()

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Dataset information
print("Dataset Information:")
df.info()

print("\n" + "="*50)
print("Statistical Summary:")
df.describe()

## 2. Data Cleaning & Preprocessing

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor()

# Check for missing values
missing_values = preprocessor.check_missing_values(df)
if len(missing_values) > 0:
    print("Missing values detected:")
    print(missing_values)
else:
    print("No missing values found!")

In [ ]:
# Handle missing values if any
df_clean = preprocessor.handle_missing_values(df, strategy='drop')

# Check churn distribution
print("\nChurn Distribution:")
print(df_clean['Exited'].value_counts())
print(f"\nChurn Rate: {df_clean['Exited'].mean() * 100:.2f}%")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Initialize analytics and visualizer
analytics = ChurnAnalytics(df_clean)
visualizer = ChurnVisualizer(df_clean)

# Calculate overall KPIs
kpis = analytics.calculate_overall_kpis()
print("Key Performance Indicators:")
for key, value in kpis.items():
    print(f"  {key}: {value}")

In [ ]:
# Churn distribution
fig = visualizer.plot_churn_distribution(save=True)
fig.show()

In [ ]:
# Geographic analysis
geo_analysis = analytics.analyze_churn_by_geography()
print("\nGeographic Churn Analysis:")
print(geo_analysis)

fig = visualizer.plot_churn_by_geography(save=True)
fig.show()

In [ ]:
# Demographic analysis
demo_analysis = analytics.analyze_churn_by_demographics()

print("\nChurn by Age Group:")
print(demo_analysis['age'])

print("\nChurn by Gender:")
print(demo_analysis['gender'])

print("\nChurn by Tenure:")
print(demo_analysis['tenure'])

fig = visualizer.plot_churn_by_demographics(save=True)
fig.show()

In [ ]:
# High-value customer analysis
high_value_analysis = analytics.analyze_high_value_churn(balance_threshold=100000)
print("\nHigh-Value Customer Analysis:")
print(high_value_analysis)

In [ ]:
# Engagement analysis
engagement_analysis = analytics.analyze_churn_by_engagement()
print("\nEngagement vs Churn:")
print(engagement_analysis)

fig = visualizer.plot_engagement_analysis(save=True)
fig.show()

In [ ]:
# Product analysis
fig = visualizer.plot_products_vs_churn(save=True)
fig.show()

fig = visualizer.plot_churn_by_tenure(save=True)
fig.show()

In [ ]:
# Correlation analysis
fig = visualizer.plot_correlation_heatmap(save=True)
fig.show()

## 4. Feature Engineering

In [ ]:
# Initialize feature engineer
feature_engineer = FeatureEngineer()

# Create all engineered features
df_engineered = feature_engineer.create_all_features(df_clean)

print(f"\nOriginal features: {len(df_clean.columns)}")
print(f"Features after engineering: {len(df_engineered.columns)}")
print(f"\nNew features created: {feature_engineer.feature_list}")

In [ ]:
# Feature importance (correlation-based)
feature_importance = feature_engineer.get_feature_importance_data(df_engineered)
print("\nTop 15 Features by Correlation with Churn:")
print(feature_importance.head(15))

## 5. Customer Segmentation

In [ ]:
# Find optimal number of clusters
elbow_results = perform_optimal_clustering(df_clean, max_clusters=8)

# Plot elbow curve
plt.figure(figsize=(10, 6))
plt.plot(elbow_results['n_clusters'], elbow_results['inertia'], marker='o', linewidth=2)
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Perform segmentation
segmenter = CustomerSegmentation(n_clusters=4)
df_segmented = segmenter.fit_segments(df_clean, method='standard')

print("\nSegment Distribution:")
print(df_segmented['Segment_Name'].value_counts())

In [ ]:
# Segment profiles
print("\nSegment Profiles:")
print(segmenter.segment_profiles)

In [ ]:
# Analyze churn by segment
segment_churn = segmenter.analyze_segment_churn(df_segmented)
print("\nChurn Analysis by Segment:")
print(segment_churn)

In [ ]:
# Segment by geography
geo_segment = segmenter.analyze_segment_by_geography(df_segmented)
if geo_segment is not None:
    print("\nSegment Distribution by Geography:")
    print(geo_segment)

In [ ]:
# Segment recommendations
recommendations = segmenter.get_segment_recommendations(df_segmented)
print("\nRecommendations by Segment:")
for segment, recs in recommendations.items():
    print(f"\n{segment}:")
    for rec in recs:
        print(f"  - {rec}")

In [ ]:
# Visualize segments
visualizer_seg = ChurnVisualizer(df_segmented)
fig = visualizer_seg.plot_segment_distribution(save=True)
fig.show()

In [ ]:
# Save segmented data
save_processed_data(df_segmented, 'cleaned_data.csv')
segmenter.save_model()
print("Segmented data and model saved!")

## 6. Churn Prediction Modeling

In [ ]:
# Prepare data for modeling
df_model = df_segmented.copy()

# Encode categorical variables
df_model_encoded = preprocessor.encode_categorical(
    df_model, 
    columns=['Geography', 'Gender'],
    method='onehot'
)

# Drop categorical features that were encoded
categorical_features = ['Age_Group', 'Balance_Category', 'Tenure_Group', 
                       'Credit_Category', 'Segment_Name']
df_model_encoded = df_model_encoded.drop(
    columns=[col for col in categorical_features if col in df_model_encoded.columns],
    errors='ignore'
)

print(f"Features for modeling: {df_model_encoded.shape[1]}")

In [ ]:
# Compare different models
predictor = ChurnPredictor(model_type='random_forest')
X, y = predictor.prepare_features(df_model_encoded)

# Split data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {len(X_train)}")
print(f"Test set: {len(X_test)}")
print(f"\nChurn distribution in training: {y_train.value_counts().to_dict()}")

In [ ]:
# Compare model performance
model_comparison = compare_models(X_train, y_train, X_test, y_test)
print("\nModel Comparison Results:")
print(model_comparison)

In [ ]:
# Train best model (usually Random Forest or Gradient Boosting)
best_model = ChurnPredictor(model_type='gradient_boosting')
best_model.train(X_train, y_train, use_smote=True)

# Evaluate
metrics = best_model.evaluate(X_test, y_test)

In [ ]:
# Feature importance
feature_importance = best_model.get_feature_importance()
print("\nTop 15 Important Features:")
print(feature_importance.head(15))

# Visualize
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['Importance'])
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Save best model
best_model.save_model()
print("Model saved successfully!")

## 7. Key Insights and Patterns

In [ ]:
# Generate comprehensive insights
analytics_final = ChurnAnalytics(df_segmented)
insights = analytics_final.identify_churn_patterns()

print("\n" + "="*60)
print("KEY INSIGHTS - Customer Churn Patterns")
print("="*60)

for i, insight in enumerate(insights, 1):
    print(f"\n{i}. {insight}")

In [ ]:
# Churned vs Retained comparison
comparison = analytics_final.compare_churned_vs_retained()
print("\n" + "="*60)
print("CHURNED vs RETAINED CUSTOMER COMPARISON")
print("="*60)
print(comparison)

In [ ]:
# Generate comprehensive summary
summary = analytics_final.generate_kpi_summary()
print("\n" + "="*60)
print("COMPREHENSIVE KPI SUMMARY")
print("="*60)
print(f"\nOverall KPIs: {summary['overall']}")
print(f"\nKey Insights: {summary['insights']}")

## 8. Export All Results

In [ ]:
# Export analysis reports
report_path = analytics_final.export_analysis_report()
print(f"Analysis reports exported to: {report_path}")

In [ ]:
# Generate all visualizations
visualizer_final = ChurnVisualizer(df_segmented)
visualizer_final.create_all_eda_plots()
print("All visualizations generated!")

## Conclusion

This notebook has provided a comprehensive analysis of customer churn in European banking. Key findings include:

1. **Geographic Patterns**: Churn varies significantly by country
2. **Demographic Insights**: Age and tenure are strong predictors
3. **Engagement Critical**: Inactive members show much higher churn
4. **Product Cross-Sell**: Multi-product customers are more loyal
5. **High-Value Risk**: Premium customers require targeted retention
6. **Segmentation**: Four distinct customer segments with varying risk profiles

Next steps:
- Launch the Streamlit dashboard for interactive analysis
- Implement targeted retention campaigns by segment
- Monitor KPIs and adjust strategies based on results